# Python对象的比较、拷贝

## 学习目标

- 理解 == 和 is 操作符的区别
- 掌握浅拷贝和深度拷贝的原理和应用场景
- 了解Python对象比较和拷贝的常见陷阱
- 学会在实际编程中正确选择比较和拷贝方式

## 一、== VS is

### 核心概念

- **== 操作符**：比较对象之间的值是否相等
- **is 操作符**：比较对象的身份标识是否相等（是否指向同一个内存地址）

在Python中，每个对象的身份标识都能通过函数 id(object) 获得。

In [2]:
# 示例1：基本类型的比较
a = 10
b = 10

print(f"a == b: {a == b}")
print(f"a is b: {a is b}")
print(f"id(a): {id(a)}")
print(f"id(b): {id(b)}")

a == b: True
a is b: True
id(a): 140714030355656
id(b): 140714030355656


### 整数缓存机制

**重要提醒**：对于整型数字，a is b 为 True 的结论只适用于 **-5 到 256** 范围内的数字。

Python为了性能优化，会对这个范围内的整型维持一个数组（缓存池）。每次创建这个范围内的整型数字时，Python会从数组中返回相对应的引用，而不是重新开辟内存空间。

In [3]:
# 示例2：超出缓存范围的整数
a = 257
b = 257

print(f"a == b: {a == b}")
print(f"a is b: {a is b}")
print(f"id(a): {id(a)}")
print(f"id(b): {id(b)}")

a == b: True
a is b: False
id(a): 2115390745744
id(b): 2115390739184


### 实际应用建议

1. **比较变量值**：通常使用 ==（更关心值是否相等）
2. **比较单例（singleton）**：通常使用 is（如检查是否为 None）
3. **性能考虑**：is 操作符速度优于 ==，因为 is 不能被重载，只需比较ID；而 == 会调用 __eq__() 方法

In [4]:
# 示例3：检查 None 的正确方式
a = None
b = [1, 2, 3]

if a is None:
    print("a is None")

if b is not None:
    print("b is not None")

# 不推荐的方式（但也能工作）
if a == None:
    print("a == None (not recommended)")

a is None
b is not None
a == None (not recommended)


### 注意事项：不可变对象的可变性

即使对于不可变（immutable）的变量，如果用 == 或 is 比较过，结果也可能改变！

**原因**：元组是不可变的，但元组可以嵌套可变对象（如列表）。修改元组中的可变元素，元组本身就会改变。

In [6]:
# 示例4：元组的"可变性"
t1 = (1, 2, [3, 4])
t2 = (1, 2, [3, 4])

print(f"Initial: t1 == t2: {t1 == t2}")

# 修改元组中的列表（可变对象）
t1[-1].append(5)
print(f"After modification: t1: {t1}")
print(f"After modification: t1 == t2: {t1 == t2}")

Initial: t1 == t2: True
t1(1, 2, [3, 4, 5])
After modification: t1: (1, 2, [3, 4, 5])
After modification: t1 == t2: False


## 二、浅拷贝和深度拷贝

### 浅拷贝（Shallow Copy）

**定义**：重新分配一块内存，创建一个新的对象，里面的元素是原对象中子对象的**引用**。

**常见方法**：
1. 使用数据类型本身的构造器：list(l1)、set(s1)
2. 切片操作符：l1[:]
3. copy.copy() 函数

In [7]:
# 示例5：浅拷贝的三种方法
import copy

# 方法1：使用构造器
l1 = [1, 2, 3]
l2 = list(l1)
print(f"Method 1 - l1 == l2: {l1 == l2}")
print(f"Method 1 - l1 is l2: {l1 is l2}")

# 方法2：切片操作符
l3 = l1[:]
print(f"\nMethod 2 - l1 == l3: {l1 == l3}")
print(f"Method 2 - l1 is l3: {l1 is l3}")

# 方法3：copy.copy()
l4 = copy.copy(l1)
print(f"\nMethod 3 - l1 == l4: {l1 == l4}")
print(f"Method 3 - l1 is l4: {l1 is l4}")

Method 1 - l1 == l2: True
Method 1 - l1 is l2: False

Method 2 - l1 == l3: True
Method 2 - l1 is l3: False

Method 3 - l1 == l4: True
Method 3 - l1 is l4: False


### 特殊案例：元组

对于元组，使用 tuple() 或切片操作符 [:]，不会创建浅拷贝，而是返回指向相同元组的引用。

In [8]:
# 示例6：元组的特殊行为
t1 = (1, 2, 3)
t2 = tuple(t1)
t3 = t1[:]

print(f"t1 == t2: {t1 == t2}")
print(f"t1 is t2: {t1 is t2}")
print(f"t1 is t3: {t1 is t3}")

print(f"\nid(t1): {id(t1)}")
print(f"id(t2): {id(t2)}")
print(f"id(t3): {id(t3)}")

t1 == t2: True
t1 is t2: True
t1 is t3: True

id(t1): 2115392098112
id(t2): 2115392098112
id(t3): 2115392098112


### 浅拷贝的副作用

**关键问题**：如果原对象中的元素是可变的，浅拷贝中的元素是对原对象元素的引用，因此修改其中一个会影响另一个！

In [9]:
# 示例7：浅拷贝的副作用
l1 = [[1, 2], (30, 40)]
l2 = list(l1)

print(f"Initial l1: {l1}")
print(f"Initial l2: {l2}")
print()

# 1. 对l1整体添加元素 - 不影响l2
l1.append(100)
print(f"After l1.append(100):")
print(f"l1: {l1}")
print(f"l2: {l2}")
print()

# 2. 修改l1中的可变对象（列表）- 会影响l2！
l1[0].append(3)
print(f"After l1[0].append(3):")
print(f"l1: {l1}")
print(f"l2: {l2}")
print()

# 3. 修改l1中的不可变对象（元组）- 创建新对象，不影响l2
l1[1] += (50, 60)
print(f"After l1[1] += (50, 60):")
print(f"l1: {l1}")
print(f"l2: {l2}")

Initial l1: [[1, 2], (30, 40)]
Initial l2: [[1, 2], (30, 40)]

After l1.append(100):
l1: [[1, 2], (30, 40), 100]
l2: [[1, 2], (30, 40)]

After l1[0].append(3):
l1: [[1, 2, 3], (30, 40), 100]
l2: [[1, 2, 3], (30, 40)]

After l1[1] += (50, 60):
l1: [[1, 2, 3], (30, 40, 50, 60), 100]
l2: [[1, 2, 3], (30, 40)]


### 深度拷贝（Deep Copy）

**定义**：重新分配一块内存，创建一个新的对象，并且将原对象中的元素，以递归的方式，通过创建新的子对象拷贝到新对象中。

**结果**：新对象和原对象没有任何关联。

**实现**：使用 copy.deepcopy() 函数

In [10]:
# 示例8：深度拷贝
import copy

l1 = [[1, 2], (30, 40)]
l2 = copy.deepcopy(l1)

print(f"Initial l1: {l1}")
print(f"Initial l2: {l2}")
print(f"l1 is l2: {l1 is l2}")
print(f"l1[0] is l2[0]: {l1[0] is l2[0]}")
print()

# 修改l1的任何元素，都不会影响l2
l1.append(100)
l1[0].append(3)

print(f"After modification:")
print(f"l1: {l1}")
print(f"l2: {l2}")

Initial l1: [[1, 2], (30, 40)]
Initial l2: [[1, 2], (30, 40)]
l1 is l2: False
l1[0] is l2[0]: False

After modification:
l1: [[1, 2, 3], (30, 40), 100]
l2: [[1, 2], (30, 40)]


### 深度拷贝的无限递归问题

**问题**：如果被拷贝对象中存在指向自身的引用，深度拷贝可能会导致无限递归。

**解决**：Python的 copy.deepcopy() 中会维护一个字典，记录已经拷贝的对象及其ID。如果字典里已经存储了将要拷贝的对象，则会直接从字典返回，避免无限递归。

**关键结论**：deepcopy 对不可变的"基础类型"（如小整数）不会创建副本——因为它们本身不可变，复用同一对象完全安全。只有遇到可变对象或需要递归拷贝的容器时，deepcopy 才会真正创建新对象。

In [11]:
# 示例9：深度拷贝处理无限递归
import copy

x = [1]
x.append(x)
print(f"x: {x}")

y = copy.deepcopy(x)
print(f"y: {y}")

# 验证x和y是独立的
print(f"x is y: {x is y}")
print(f"x[0] is y[0]: {x[0] is y[0]}")
print(f"x[1] is y[1]: {x[1] is y[1]}")

x: [1, [...]]
y: [1, [...]]
x is y: False
x[0] is y[0]: True
x[1] is y[1]: False


## 三、总结

### 核心要点

1. **比较操作符**
   - ==：比较对象的值是否相等，会调用 __eq__() 方法
   - is：比较对象的ID是否相等，效率更高

2. **拷贝操作**
   - **浅拷贝**：创建新对象，但元素是原对象子对象的引用（可能存在副作用）
   - **深度拷贝**：创建新对象，递归拷贝所有子对象（完全独立，但可能较慢）

3. **实际应用**
   - 比较值用 ==，比较单例用 is
   - 如果对象包含可变元素，且需要独立副本，使用深度拷贝
   - 如果对象只包含不可变元素，浅拷贝就足够了

## 四、练习与自测

### 练习题1：比较操作符的应用

**题目**：写出下面代码的输出结果，并解释原因。

```python
a = 256
b = 256
c = 257
d = 257

print(a is b) # True
print(c is d) #False
print(a == b) #True
print(c == d) #True
```

**参考答案**：
- `a is b` → `True`（256在缓存范围内）
- `c is d` → `False`（257超出缓存范围，创建了两个不同对象）
- `a == b` → `True`（值相等）
- `c == d` → `True`（值相等）

In [1]:
# 练习1：验证你的答案
a = 256
b = 256
c = 257
d = 257

print(f"a is b: {a is b}")
print(f"c is d: {c is d}")
print(f"a == b: {a == b}")
print(f"c == d: {c == d}")

a is b: True
c is d: False
a == b: True
c == d: True


### 练习题2：浅拷贝 vs 深度拷贝

**题目**：有如下代码，请回答：
1. 修改 `l1[0][0] = 100` 后，`l2` 的值会变化吗？
2. 如果想让 `l1` 和 `l2` 完全独立，应该如何修改代码？

```python
import copy

l1 = [[1, 2], [3, 4]]
l2 = copy.copy(l1)
```

**参考答案**：
1. **会变化**。因为 `l2 = copy.copy(l1)` 是浅拷贝，`l1[0]` 和 `l2[0]` 指向同一个列表对象，修改其中一个会影响另一个。
2. **使用深度拷贝**：`l2 = copy.deepcopy(l1)`，这样 `l1` 和 `l2` 完全独立，修改任意一个都不会影响另一个。

In [12]:
# 练习2：验证浅拷贝的副作用
import copy

l1 = [[1, 2], [3, 4]]
l2 = copy.copy(l1)

print(f"Before modification:")
print(f"l1: {l1}")
print(f"l2: {l2}")

# 修改l1[0][0]
l1[0][0] = 100

print(f"\nAfter l1[0][0] = 100:")
print(f"l1: {l1}")
print(f"l2: {l2}")

# 使用深度拷贝解决问题
print(f"\n--- Using deep copy ---")
l3 = [[1, 2], [3, 4]]
l4 = copy.deepcopy(l3)

l3[0][0] = 100
print(f"After l3[0][0] = 100:")
print(f"l3: {l3}")
print(f"l4: {l4}")

Before modification:
l1: [[1, 2], [3, 4]]
l2: [[1, 2], [3, 4]]

After l1[0][0] = 100:
l1: [[100, 2], [3, 4]]
l2: [[100, 2], [3, 4]]

--- Using deep copy ---
After l3[0][0] = 100:
l3: [[100, 2], [3, 4]]
l4: [[1, 2], [3, 4]]


## 五、知识总结

### 重点记忆

1. **== vs is**
   - == 比较值，is 比较ID
   - 整数缓存范围：-5 ~ 256
   - 检查 None 用 is None

2. **浅拷贝**
   - 创建新对象，元素是原对象子对象的引用
   - 方法：list()、[:]、copy.copy()
   - 风险：修改可变子对象会影响所有拷贝

3. **深度拷贝**
   - 递归拷贝所有子对象，完全独立
   - 方法：copy.deepcopy()
   - 优势：无副作用
   - 注意：会处理无限递归问题

### 实践建议

- 简单场景（不可变对象）：使用浅拷贝或直接使用赋值
- 复杂场景（包含可变对象）：使用深度拷贝确保安全性
- 性能考虑：如果确定对象不可变，优先使用浅拷贝（更快）

## 思考题（课程作业）

### 题目

使用深度拷贝拷贝一个无限嵌套的列表后，用等于操作符 == 进行比较，输出会是什么？是 True、False 还是其他？为什么？

```python
import copy

x = [1]
x.append(x)
y = copy.deepcopy(x)

# 以下命令的输出是？
print(x == y)
```

### 提示

思考以下几个问题：
1. x 和 y 的结构是什么？
2. == 操作符在比较列表时是如何工作的？
3. 无限嵌套的结构会导致什么问题？

**建议**：先自己思考，然后实际运行代码验证你的猜想！

In [ ]:
import copy

x = [1]
x.append(x)
y = copy.deepcopy(x)

print(x == y)


**解释**

RecursionError = 递归错误（递归超出最大深度）

Python 中默认的最大递归深度是 1000 层，超过就会抛出这个异常。

RecursionError: maximum recursion depth exceeded in comparison

     ↑                ↑                                      ↑
  递归错误              最大递归深度被超越                          在比较操作中

